# Static Validation

**Purpose:** Validate thermal model calculations across building archetypes — energy
performance range (DPE A-G), consumption by construction period, and static cost-efficiency.

**Project imports:** `project.thermal` (conventional_energy_3uses, certificate, HDD, CONVERSION),
`project.input.param` (generic_input), `project.model` (get_inputs).

**Inputs:** Building archetypes, efficiency data, performance_insulation.

**Outputs:** Validation tables (DPE range check), consumption by period, static analysis output.

**Consolidates:** Previously split across `validation_thermal_function.ipynb`,
`building_consumption.ipynb`, and `cost_efficiency.ipynb`.

## 1. Setup

In [ ]:
import pandas as pd

from project.thermal import conventional_energy_3uses, certificate, HDD, CONVERSION, heating_consumption
from project.input.param import generic_input
from project.model import get_inputs

In [ ]:
ratio_surface = generic_input['ratio_surface']

In [ ]:
output = get_inputs(variables=['buildings', 'efficiency', 'performance_insulation'],
                    building_stock='project/input/stock/buildingstock_sdes2018_medium_3.csv')
buildings, eff, performance_insulation = output['buildings'], output['efficiency'], output['performance_insulation']

## 2. Thermal function validation

Tests the dispersion of space heating consumption and energy performance for best and worst
building archetypes. The performance should range from A to G to address French policy.

### Defining boundary archetypes

In [ ]:
worst_insulation = {i: max(buildings.stock.index.get_level_values(i))
                    for i in ['Wall', 'Roof', 'Floor', 'Windows']}
worst_insulation.update({'Ventilation': 0.8})
print('Worst:', worst_insulation)

best_insulation = {i: min(buildings.stock.index.get_level_values(i))
                   for i in ['Wall', 'Roof', 'Floor', 'Windows']}
print('Best:', best_insulation)

performance_insulation.update({'Ventilation': 0.4})
print('Performance target:', performance_insulation)

In [ ]:
# Wall, Roof, Floor, Windows, Air rate (ventilation), Heating system
archetypes = {
    'Worst electricity': [i for i in worst_insulation.values()] + ['Electricity-Performance boiler'],
    'Best electricity water': [i for i in performance_insulation.values()] + ['Electricity-Heat pump water'],
    'Best electricity air': [i for i in performance_insulation.values()] + ['Electricity-Heat pump air'],
    'Worst gas': [i for i in worst_insulation.values()] + ['Natural gas-Standard boiler'],
    'Best gas': [i for i in performance_insulation.values()] + ['Natural gas-Performance boiler'],
}

archetypes = pd.DataFrame(archetypes,
                          index=['Wall', 'Roof', 'Floor', 'Windows', 'Ventilation', 'Heating system']).T
archetypes['Energy'] = archetypes['Heating system'].str.split('-').str[0].rename('Energy')

archetypes = pd.concat([archetypes, archetypes],
                       keys=['Single-family', 'Multi-family'], names=['Housing type'])
efficiency = archetypes['Heating system'].replace(eff.to_dict()).set_axis(archetypes.index)
archetypes = pd.concat((archetypes, efficiency.rename('Efficiency')), axis=1)
archetypes = archetypes.set_index(['Heating system', 'Energy'], append=True)
archetypes = archetypes.droplevel(None)
archetypes = archetypes.astype({
    'Wall': 'float', 'Roof': 'float', 'Floor': 'float',
    'Windows': 'float', 'Ventilation': 'float', 'Efficiency': 'float'
})
archetypes

### Energy dispersion without ventilation

In [ ]:
dpe, energy_primary = conventional_energy_3uses(
    archetypes['Wall'], archetypes['Floor'], archetypes['Roof'], archetypes['Windows'],
    ratio_surface, archetypes['Efficiency'], archetypes.index
)
result = pd.concat((archetypes, dpe.rename('Performance'), energy_primary.rename('Energy')),
                   axis=1).sort_index()
result

- The performance ranges from A to G, which is the necessary range for French policy.
- Worst performing fossil fuel multi-family buildings are not G because of using average
  loss surface for each component. First and last floors would get more, others less.
  Archetypes represent an average apartment, centering distribution around the mean.

### Energy dispersion with ventilation

In [ ]:
dpe, energy_primary = conventional_energy_3uses(
    archetypes['Wall'], archetypes['Floor'], archetypes['Roof'], archetypes['Windows'],
    ratio_surface, archetypes['Efficiency'], archetypes.index,
    air_rate=archetypes['Ventilation']
)
result = pd.concat((archetypes, dpe.rename('Performance'), energy_primary.rename('Energy')),
                   axis=1).sort_index()
result

Adding ventilation does not significantly change the dispersion.

### Testing with rounded values

In [ ]:
buildings_round = archetypes.copy()
for col in ['Wall', 'Floor', 'Roof', 'Windows', 'Efficiency']:
    buildings_round[col] = buildings_round[col].astype('float').round(1)

dpe, energy_primary = conventional_energy_3uses(
    buildings_round['Wall'], buildings_round['Floor'], buildings_round['Roof'],
    buildings_round['Windows'], ratio_surface, buildings_round['Efficiency'],
    buildings_round.index, air_rate=archetypes['Ventilation']
)
result = pd.concat((buildings_round, dpe, energy_primary), axis=1).sort_index()
result

## 3. Consumption by construction period

Tests heating consumption for simplified building archetypes grouped by construction period.

In [ ]:
DHH = HDD  # alias for compatibility

buildings_simple = {
    'Renovated': [0.2, 0.2, 0.2, 1.3, 0.8],
    'Old': [2, 2, 2, 4, 0.7],
    'New': [0.1, 0.1, 0.1, 1.2, 3]
}

buildings_simple = pd.DataFrame(buildings_simple,
                                index=pd.Index(['Wall', 'Floor', 'Roof', 'Windows', 'Efficiency'])).T
buildings_simple = pd.concat([buildings_simple] * 2,
                             keys=['Single-family', 'Multi-family'], names=['Housing type'])
buildings_simple

In [ ]:
conventional_energy_3uses(
    buildings_simple['Wall'], buildings_simple['Floor'], buildings_simple['Roof'],
    buildings_simple['Windows'], buildings_simple['Efficiency'], ratio_surface, HDD
)

### Consumption by detailed construction period

In [ ]:
buildings_period = {
    '1914': [1.35, 1.5, 2.3, 4.8, 0.8],
    '1948': [1.35, 2.5, 1.7, 4.8, 0.8],
    '1967': [2.42, 1.8, 2.8, 2.6, 0.8],
    '1974': [1.35, 2.8, 2.5, 2.6, 0.8],
    '1981': [0.57, 0.61, 0.46, 2.8, 0.8],
    '1989': [0.32, 0.42, 1.25, 2.6, 0.8],
    '1999': [0.23, 0.36, 0.42, 2.6, 0.8],
    '2005': [0.19, 0.33, 0.42, 1.8, 0.8],
    '2012': [0.239, 0.29, 0.18, 1.4, 0.8],
    '2020': [0.17, 0.21, 0.21, 1.4, 0.8]
}

buildings_period = pd.DataFrame(buildings_period,
                                index=pd.Index(['Wall', 'Floor', 'Roof', 'Windows', 'Efficiency'])).T
buildings_period = pd.concat([buildings_period] * 2,
                             keys=['Single-family', 'Multi-family'], names=['Housing type'])

heating_consumption(
    buildings_period['Wall'], buildings_period['Floor'], buildings_period['Roof'],
    buildings_period['Windows'], buildings_period['Efficiency'], ratio_surface, DHH
)

## 4. Static cost-efficiency analysis

Runs the full `make_static_analysis()` method which computes profitability,
subsidy requirements, and distributional effects across the building stock.

In [ ]:
inputs = get_inputs(variables=['buildings', 'energy_prices', 'cost_insulation',
                               'cost_heater', 'carbon_value_kwh', 'carbon_emission',
                               'health_cost', 'implicit_discount_rate'])

year = 2018
buildings_full = inputs['buildings']

buildings_full.make_static_analysis(
    inputs['cost_insulation'],
    inputs['cost_heater'],
    inputs['energy_prices'].loc[year, :],
    0.05,
    inputs['implicit_discount_rate'],
    inputs['health_cost'],
    inputs['carbon_value_kwh'].loc[year, :],
    inputs['carbon_emission'].loc[year, :],
    path_out='output'
)